In [1]:
import numpy as np
from joblib import load

import torch
from torch.utils.data import Dataset, DataLoader, get_worker_info

from pathlib import Path

import cProfile

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme()
%matplotlib inline

In [3]:
class JoblibDataset(Dataset):  # TODO: Dictionary outputs!!!

    def __init__(self, file_paths, subseq_len, keys=['mhr', 'ece', 'co2']):
        super().__init__()
        self.file_paths = file_paths
        self.subseq_len = subseq_len
        self.keys = keys
        self.data_shapes = None

        # Don't open the files here; just build an index (subseq_info) of what subsequences exist.
        self._opened_files = None  # This remains None until worker_init_fn calls .worker_init()

        self.subseq_info = []
        # Build an index of all non-overlapping chunks across all files by reading shapes only.
        for f_idx, fp in enumerate(self.file_paths):
            # Load enough info to get n_samples
            data_dict = load(fp, mmap_mode='r')
            if self.data_shapes is None:
                self.data_shapes = {key: data_dict[key].shape[:-1] + (subseq_len, ) for key in keys}
            all_n_samples = [data_dict[key].shape[-1] for key in keys]
            n_samples = np.min(all_n_samples)
            # Drop the reference to close the file handle
            del data_dict

            # If we can fit at least one subsequence
            if self.subseq_len == -1:
                self.subseq_info.append((f_idx, 0))
                continue
            if n_samples >= self.subseq_len:
                n_chunks = n_samples // self.subseq_len
                for chunk_idx in range(n_chunks):
                    start_samp = chunk_idx * self.subseq_len
                    self.subseq_info.append((f_idx, start_samp))

        # The total number of subsequences across all files
        self.total_subseqs = len(self.subseq_info)

    def __len__(self):
        return self.total_subseqs

    def worker_init(self):
        self._opened_files = []
        for fp in self.file_paths:
            data_dict = load(fp, mmap_mode='r')
            filtered_dict = {key: data_dict[key] for key in self.keys}
            self._opened_files.append(filtered_dict)

    def _load_single_sample(self, idx):
        """
        Load a single sample from the opened files.
        This is used in __getitem__ and __getitems__.
        """
        if self._opened_files is None:
            self.worker_init()

        file_idx, start_samp = self.subseq_info[idx]

        data_dict = self._opened_files[file_idx]

        # Slicing out a non-overlapping subsequence
        end_samp = start_samp + self.subseq_len
        """
        inputs = dict()
        for key in self.keys:
            inputs[key] = torch.from_numpy(data_dict[key][..., start_samp:end_samp])
        """
        inputs = [torch.from_numpy(data_dict[key][..., start_samp:end_samp]) for key in self.keys]

        return inputs

    def __getitem__(self, idx):
        data = self._load_single_sample(idx)
        return {key: val for key, val in zip(self.keys, data)}

    def __getitems__(self, indices):
        """
        Optimized batch loading using __getitems__
        This method is called automatically when using BatchSampler
        """
        batch_size = len(indices)
        
        # Load all samples using the existing __getitem__ logic
        samples = [self._load_single_sample(idx) for idx in indices]
        """
        # Find C dimensions for this batch
        C = [sample.shape[0] for sample in samples[0]]
        
        # Get F, T from first sample (assuming they're consistent)
        F, T = samples[0][0].shape[1], samples[0][0].shape[2]
        """
        batch_tensors = [
            torch.empty((batch_size, *shape), dtype=torch.float32) for shape in self.data_shapes.values()
        ]
        
        for i, sample in enumerate(samples):
            for j, s in enumerate(sample):
                batch_tensors[j][i] = s
        return {key: val for key, val in zip(self.keys, batch_tensors)}

In [4]:
def my_worker_init_fn(worker_id):
    # This is called in each worker process once
    worker_info = get_worker_info()
    dataset = worker_info.dataset
    dataset.worker_init()  # Actually open the joblib files in this worker

In [5]:
def collate_fn(batch):
    """
    Custom collate function for __getitems__ approach
    """
    # __getitems__ returns a list with one element containing batched tensors
    return batch

In [6]:
%%time
# training_files_dir = Path('/scratch/gpfs/EKOLEMEN/hackathon/foundation25/spectrogram/')
training_files_dir = Path('./')
training_set = JoblibDataset(list(training_files_dir.glob("*.joblib"))[:500],
                             subseq_len=256)

train_loader = DataLoader(
    training_set,
    batch_size=16,
    # pin_memory=True,
    shuffle=True,
    num_workers=0,
    # persistent_workers=True,
    collate_fn=collate_fn,
    worker_init_fn=my_worker_init_fn
)

# train_loader = DataLoader(training_set, batch_size=16, pin_memory=True, shuffle=False,
#                           num_workers=0, worker_init_fn=my_worker_init_fn)

CPU times: total: 0 ns
Wall time: 0 ns


In [7]:
def iterate_train_loader():
    for batch_data in train_loader:
        for data in batch_data:
            pass

In [8]:
profiler = cProfile.Profile()
profiler.enable()
iterate_train_loader()  # Call the function to profile
profiler.disable()
profiler.print_stats(sort='cumulative') # Sort by cumulative time

C:\Users\admin\AppData\Local\Temp\ipykernel_114504\3480888288.py:67: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  inputs = [torch.from_numpy(data_dict[key][..., start_samp:end_samp]) for key in self.keys]


         15114 function calls (14979 primitive calls) in 1.379 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        8    0.000    0.000    1.178    0.147 dataloader.py:728(__next__)
        8    0.000    0.000    1.177    0.147 dataloader.py:787(_next_data)
        7    0.000    0.000    1.177    0.168 fetch.py:47(fetch)
        7    1.175    0.168    1.177    0.168 3480888288.py:75(__getitems__)
        6    0.000    0.000    0.200    0.033 base_events.py:1908(_run_once)
        6    0.000    0.000    0.199    0.033 selectors.py:319(select)
        6    0.000    0.000    0.199    0.033 selectors.py:313(_select)
       97    0.000    0.000    0.013    0.000 3480888288.py:48(_load_single_sample)
        1    0.000    0.000    0.011    0.011 3480888288.py:41(worker_init)
        4    0.000    0.000    0.011    0.003 numpy_pickle.py:674(load)
        4    0.000    0.000    0.010    0.003 numpy_pickle.py:613(_unpickle)
   

In [9]:
profiler = cProfile.Profile()
profiler.enable()
iterate_train_loader()  # Call the function to profile
profiler.disable()
profiler.print_stats(sort='cumulative') # Sort by cumulative time

         2375 function calls (2335 primitive calls) in 0.199 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.191    0.095 interactiveshell.py:3663(run_code)
        2    0.000    0.000    0.191    0.095 {built-in method builtins.exec}
        1    0.000    0.000    0.191    0.191 2121102005.py:1(<module>)
        8    0.011    0.001    0.165    0.021 dataloader.py:728(__next__)
        8    0.000    0.000    0.152    0.019 dataloader.py:787(_next_data)
        7    0.000    0.000    0.152    0.022 fetch.py:47(fetch)
        7    0.151    0.022    0.152    0.022 3480888288.py:75(__getitems__)
        1    0.026    0.026    0.031    0.031 2699206824.py:1(iterate_train_loader)
        2    0.007    0.004    0.007    0.004 {method '__exit__' of 'sqlite3.Connection' objects}
       97    0.000    0.000    0.001    0.000 3480888288.py:48(_load_single_sample)
      291    0.000    0.000    0.001 

In [10]:
data = next(iter(train_loader))

In [11]:
[(key, val.shape) for key, val in data.items()]

[('mhr', torch.Size([16, 8, 513, 256])),
 ('ece', torch.Size([16, 48, 513, 256])),
 ('co2', torch.Size([16, 4, 513, 256]))]